In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

#### Example 5.1 : BlackJack

In [ ]:
def codeBJState(hand, faceUp, usableAce):
    return 20 * (hand - 12) + 2 * (faceUp - 1) + usableAce

def decodeBJState(state):   
    handNorm, r = divmod(state, 20)
    faceUpNorm, usableAce = divmod(r, 2)
    hand = handNorm + 12
    dealerCard = faceUpNorm + 1
    return hand, dealerCard, usableAce

In [ ]:
def sampleBJTrajectory(policy, startingState=None, startingAction=None):
    cards = np.arange(1, 11)
    cardsProb = [1 / 13] * 9 + [4 / 13]
    randomizedPolicy = len(policy.shape) == 2
 
    # Preping up the hands
    if startingState is None:
        playerFirstCard = np.random.choice(cards, p=cardsProb)
        playerSecondCard = np.random.choice(cards, p=cardsProb)
 
        playerHand = playerFirstCard + playerSecondCard
        playerUsableAce = 1 * ((1 in [playerFirstCard, playerSecondCard]) and (playerHand + 10 <= 21))
 
        dealerFaceUpCard = np.random.choice(cards, p=cardsProb)
 
        while 10 * playerUsableAce + playerHand <= 11:
            newCard = np.random.choice(cards, p=cardsProb)
            playerHand += newCard
            playerUsableAce = 1 * ((playerUsableAce or newCard == 1) and (10 + playerHand <= 21))
 
    else:
        playerHand, dealerFaceUpCard, playerUsableAce = decodeBJState(startingState)
        playerHand -= 10 * playerUsableAce

 
    dealerFaceDownCard = np.random.choice(cards, p=cardsProb)
    dealerHand = dealerFaceUpCard + dealerFaceDownCard
    dealerUsableAce = 1 * ((1 in [dealerFaceUpCard, dealerFaceDownCard]) and (dealerHand + 10 <= 21))
 
    stateTrajectory = []
    actionTrajectory = []
 
    # Game play
    if startingAction == 0: # hit, update hand
        stateTrajectory += [startingState]
        actionTrajectory += [0]
        newCard = np.random.choice(cards, p=cardsProb)
        playerHand += newCard
        playerUsableAce = 1 * ((playerUsableAce or newCard == 1) and (10 + playerHand <= 21))
 
    if startingAction != 1:
        if 10 * playerUsableAce + playerHand <= 21:
            if randomizedPolicy:
                nextAction = np.random.choice([0, 1], p=policy[codeBJState(10 * playerUsableAce + playerHand, dealerFaceUpCard, playerUsableAce), :])
            else:
                nextAction = policy[codeBJState(10 * playerUsableAce + playerHand, dealerFaceUpCard, playerUsableAce)]
 
        while 10 * playerUsableAce + playerHand <= 21 and nextAction == 0:
            stateTrajectory += [codeBJState(10 * playerUsableAce + playerHand, dealerFaceUpCard, playerUsableAce)]
            actionTrajectory += [0]
            newCard = np.random.choice(cards, p=cardsProb)
            playerHand += newCard
            playerUsableAce = 1 * ((playerUsableAce or newCard == 1) and (10 + playerHand <= 21))
            if 10 * playerUsableAce + playerHand <= 21:
                if randomizedPolicy:
                    nextAction = np.random.choice([0, 1], p=policy[codeBJState(10 * playerUsableAce + playerHand, dealerFaceUpCard, playerUsableAce), :])
                else:
                    nextAction = policy[codeBJState(10 * playerUsableAce + playerHand, dealerFaceUpCard, playerUsableAce)]
       
    if 10 * playerUsableAce + playerHand > 21:
        reward = -1
 
    else:
        stateTrajectory += [codeBJState(10 * playerUsableAce + playerHand, dealerFaceUpCard, playerUsableAce)]
        actionTrajectory += [1]
 
        while 10 * dealerUsableAce + dealerHand <= 16:
            newCard = np.random.choice(cards, p=cardsProb)
            dealerHand += newCard
            dealerUsableAce = 1 * ((dealerUsableAce or newCard == 1) and (10 + dealerHand <= 21))
 
        if 10 * dealerUsableAce + dealerHand > 21:
            reward = 1
 
        else:
            reward = - 1 * (10 * dealerUsableAce + dealerHand > 10 * playerUsableAce + playerHand) + 1 * (10 * dealerUsableAce + dealerHand < 10 * playerUsableAce + playerHand)
 
    trajectory = np.empty((len(stateTrajectory), 3), dtype=int)
    trajectory[:, 0] = stateTrajectory
    trajectory[:, 1] = actionTrajectory
    trajectory[:, 2] = [0] * (len(stateTrajectory) - 1) + [reward]
 
    return trajectory

In [ ]:
def plotPolicyBJ(policy):
    x = np.arange(1, 11)
    y = np.arange(12, 22)
 
    fig = make_subplots(rows=1, cols=2,
    subplot_titles=["no usable ace", "usable ace"])
 
    Xy1, Yy1 = np.meshgrid(x, y[1:])
    Xy_1, Yy_1 = np.meshgrid(x, y[:-1])
    Xx1, Yx1 = np.meshgrid(x[1:], y)
    Xx_1, Yx_1 = np.meshgrid(x[:-1], y)
    X, Y = np.meshgrid(x, y)
 
    for z in [0, 1]:
        if len(policy.shape) == 2:
            Z = policy[codeBJState(Y, X, z), 1]
            fig.add_trace(go.Heatmap(x=x, y=y, z=Z, colorbar=dict(x=0.45 if z ==0 else 1.02)), row=1, col=z+1)
 
        else:
            diff_x = policy[codeBJState(Yx1, Xx1, z)] != policy[codeBJState(Yx_1, Xx_1, z)]
            diff_y = policy[codeBJState(Yy1, Xy1, z)] != policy[codeBJState(Yy_1, Xy_1, z)]
 
            ix, iy = np.where(diff_x)
            for i, j in zip(ix, iy):
                fig.add_trace(
                    go.Scatter(
                        x=[1+j, 1+j],
                        y=[12+i-1, 12+i],
                        mode="lines",
                        line=dict(color="black"),
                        showlegend=False,
                        hoverinfo="none"
                    ),
                    row=1,
                    col=z+1,
                )
 
            ix, iy = np.where(diff_y)
            for i, j in zip(ix, iy):
                fig.add_trace(
                    go.Scatter(
                        x=[1+j-1, 1+j],
                        y=[12+i, 12+i],
                        mode="lines",
                        line=dict(color="black"),
                        showlegend=False,
                        hoverinfo="none"
                    ),
                    row=1,
                    col=z+1,
                )
 
            fig.update_xaxes(range=[0, 10], tickmode="array", tickvals=np.arange(11), ticks="inside", dtick=1, showticklabels=False, row=1, col=z+1)
 
            for i in range(11):
                fig.add_annotation(
                    x=i + 0.5,
                    y=0,
                    xref=f"x{2 if z == 1 else ''}",
                    yref=f"y{2 if z == 1 else ''} domain",
                    text=str(i + 1) if i >= 1 else "A" ,
                    showarrow=False,
                    yshift=-25,
                )
 
            fig.update_yaxes(range=[10, 21], tickvals=np.arange(10, 22), ticktext=np.arange(10, 22), ticks="inside", dtick=1, showticklabels=False, row=1, col=z+1)
               
            for i in range(10, 22):
                fig.add_annotation(
                    y=i + 0.5,
                    x=0,
                    xref=f"x{2 if z == 1 else ''}",
                    yref=f"y{2 if z == 1 else ''}",
                    text=str(i + 1),
                    showarrow=False,
                    xshift=-25,
                    row=1,
                    col=z+1,
                )
               
    fig.update_layout(showlegend=False)
    fig.show()

In [ ]:
def plotValueBJ(V):
    x = np.arange(12, 22)
    y = np.arange(1, 11)
 
    X, Y = np.meshgrid(x, y)
 
    fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'surface'}, {'type': 'surface'}]], subplot_titles=["no usable ace", "usable ace"])
    for z in [0, 1]:
        fig.add_trace(go.Surface(x=x, y=y, z=V[codeBJState(X, Y, z)], colorbar=dict(x=0.45 if z ==0 else 1.02)), row=1, col=z+1)
 
    fig.update_layout(showlegend=False)
    fig.show()

In [ ]:
# BlackJack model
nStatesBJ = 200
nActionsBJ = 2
actionMappingBJ = np.array([[0, 1] for s in range(nStatesBJ)])
legalActionsBJ = np.zeros((nStatesBJ, nActionsBJ), dtype=bool)
for state, action in enumerate(actionMappingBJ):
    legalActionsBJ[state, action] = True

modelBJ = nStatesBJ, nActionsBJ, legalActionsBJ, actionMappingBJ

In [ ]:
basePolicyBJ = np.empty(nStatesBJ, dtype=int)
for dealerFaceUpCard in range(1, 11):
    for usableAce in range(2):
        for playerHand in range(12, 20):
            basePolicyBJ[codeBJState(playerHand, dealerFaceUpCard, usableAce)] = 0
        basePolicyBJ[codeBJState(20, dealerFaceUpCard, usableAce)] = 1
        basePolicyBJ[codeBJState(21, dealerFaceUpCard, usableAce)] = 1
plotPolicyBJ(basePolicyBJ)

In [ ]:
def decodeTrajectoryBJ(trajectory):
    decodedStates = []
    for s in trajectory[:, 0]:
        decodedStates += [decodeBJState(s)]
       
    decodedActions = []
    for a in trajectory[:, 1]:
        decodedActions += ["stick" * a + "hit" * (1 - a)]
 
    decoded = np.empty(trajectory.shape, dtype=object)
    decoded[:, 0] = decodedStates
    decoded[:, 1] = decodedActions
    decoded[:, 2] = trajectory[:, 2]
 
    return decoded

In [ ]:
def monteCarloPredictionBJ(model, gamma, maxIter, vInit, policy, ES=False):
    nStates, nActions, legalActions, actionMapping = model
    V = vInit.copy()
    nVisits = np.zeros(nStates)
 
    iter = 0        
    while iter < maxIter:
        if iter % (maxIter // 5) == 0:
            print(f"Iteration {iter}")
 
        if ES:
            startingState = np.random.choice(nStates)
            startingAction = policy[startingState]
            trajectory = sampleBJTrajectory(policy, startingState=startingState, startingAction=startingAction)
        else:
            trajectory = sampleBJTrajectory(policy)
 
        gain = 0.0
        T = len(trajectory)
        for t in range(T - 1, -1, -1):
            gain = gamma * gain + trajectory[t, 2]
            currentState = trajectory[t, 0]
            nVisits[currentState] += 1
            V[currentState] = V[currentState] + (gain - V[currentState]) / nVisits[currentState]
 
        iter += 1
 
    return V

In [ ]:
nItersBJ = 500000
gammaBJ = 1
vInitBJ = np.zeros(nStatesBJ)
vBasePolicyBJ = monteCarloPredictionBJ(modelBJ, gammaBJ, nItersBJ, vInitBJ, basePolicyBJ)
plotValueBJ(vBasePolicyBJ)

#### Example 5.3 : Solving BlackJack

In [ ]:
def monteCarloControlBJ(model, gamma, maxIter, qInit, policyInit):
    nStates, nActions, legalActions, actionMapping = model
   
    Q = qInit.copy()
    policy = policyInit.copy()
    nVisits = np.zeros((nStates, nActions))
    iter = 0
 
    while iter < maxIter:
        if iter % (maxIter // 5) == 0:
            print(f"Iteration {iter}")
 
        startingState = np.random.choice(nStates)
        startingAction = np.random.choice(actionMapping[startingState])
 
        trajectory = sampleBJTrajectory(policy, startingState=startingState, startingAction=startingAction)
        
        gain = 0.0
        T = len(trajectory)
        for t in range(T - 1, -1, -1):
            gain = gamma * gain + trajectory[t, 2]
            currentState = trajectory[t, 0]
            currentAction = trajectory[t, 1]
            nVisits[currentState, currentAction] += 1
            Q[currentState, currentAction] = Q[currentState, currentAction] + (gain - Q[currentState, currentAction]) / nVisits[currentState, currentAction]
            policy[currentState] = np.argmax(Q[currentState, :])

 
        iter += 1
 
    return Q, policy

In [ ]:
nItersBJOpt = 1000000
qInitBJ = np.zeros((nStatesBJ, nActionsBJ))
qStarBJ, policyStarBJ = monteCarloControlBJ(modelBJ, gammaBJ, nItersBJOpt, qInitBJ, basePolicyBJ)
vStarBJ = qStarBJ[np.arange(nStatesBJ), policyStarBJ]

In [ ]:
plotValueBJ(vStarBJ)
plotPolicyBJ(policyStarBJ)

#### $\epsilon$-soft policy for the BlackJack problem

In [ ]:
def softMonteCarloControlBJ(model, epsilon, gamma, maxIter, qInit, policyInit):
    nStates, nActions, legalActions, actionMapping = model
   
    Q = qInit.copy()
    policy = policyInit.copy()
    nVisits = np.zeros((nStates, nActions))
    iter = 0
 
    while iter < maxIter:
        if iter % (maxIter // 5) == 0:
            print(f"Iteration {iter}")
 
        trajectory = sampleBJTrajectory(policy)

        gain = 0.0
        T = len(trajectory)
        for t in range(T - 1, -1, -1):
            gain = gamma * gain + trajectory[t, 2]
            currentState = trajectory[t, 0]
            currentAction = trajectory[t, 1]
 
            nVisits[currentState, currentAction] += 1
            Q[currentState, currentAction] = Q[currentState, currentAction] + (gain - Q[currentState, currentAction]) / nVisits[currentState, currentAction]
            aStar = np.argmax(Q[currentState, :])

            policy[currentState] = [epsilon / 2, epsilon / 2]
            policy[currentState, aStar] += 1 - epsilon
 
        iter += 1
                   
    return Q, policy

In [ ]:
epsilonBJ = 0.01
epsilonSoftPolicyBJ = 0.5 * epsilonBJ * np.ones((nStatesBJ, nActionsBJ))
epsilonSoftPolicyBJ[:, 1] += 1 - epsilonBJ
qEpsilonSoftStarBJ, policyEpsilonSoftStarBJ = softMonteCarloControlBJ(modelBJ, epsilonBJ, gammaBJ, nItersBJOpt, qInitBJ, epsilonSoftPolicyBJ)
vEpsilonSoftStarBJ = np.sum(policyEpsilonSoftStarBJ * qEpsilonSoftStarBJ, axis=1)

In [ ]:
plotValueBJ(vEpsilonSoftStarBJ)
plotPolicyBJ(policyEpsilonSoftStarBJ)

#### Example 5.4 : Off-policy Estimation of a BlackJack State Value

In [ ]:
basePolicyTestStateBJ = np.zeros((nStatesBJ, nActionsBJ), dtype=int)
basePolicyTestStateBJ[np.arange(nStatesBJ), basePolicyBJ] = 1
behaviourPolicyTestStateBJ = np.array([[0.5, 0.5]] * nStatesBJ)
testStateBJ = codeBJState(13, 2, 1)

In [ ]:
def OffPolicyFVBJ(nIters):
    Qweighted = np.zeros(2)
    Qordinary = np.zeros(2)
    ratios = np.zeros(2)
    nVisits = np.zeros(2)
    learningCurveWeighted = np.empty(nIters)
    learningCurveOrdinary = np.empty(nIters)
 
    iter = 0
    learningCurveWeighted[0] = Qweighted[0]
    learningCurveOrdinary[0] = Qordinary[0]
 
    while iter < nIters:
        firstAction = np.random.choice([0, 1])
        trajectory = sampleBJTrajectory(behaviourPolicyTestStateBJ, testStateBJ, firstAction)
        weight = 1.0
        T = len(trajectory)
        gain = trajectory[T - 1, 2]
        for t in range(T - 2 , -1, -1):
            weight *= basePolicyTestStateBJ[trajectory[t, 0], trajectory[t, 1]] / behaviourPolicyTestStateBJ[trajectory[t, 0], trajectory[t, 1]]
       
        nVisits[firstAction] += 1
        Qordinary[firstAction] += (weight * gain - Qordinary[firstAction]) / nVisits[firstAction]
 
        if weight > 0:
            ratios[firstAction] += weight
            Qweighted[firstAction] += weight * (gain - Qweighted[firstAction]) / ratios[firstAction]
 
        iter += 1
        if iter < nIters:
            learningCurveWeighted[iter] = Qweighted[0]
            learningCurveOrdinary[iter] = Qordinary[0]
 
    return learningCurveWeighted, learningCurveOrdinary

In [ ]:
def prepareExperimentsBJ(nRuns, nIters):
    weightedCurves = []
    ordinaryCurves = []
    for run in range(nRuns):
        if run % (nRuns // 10) == 0:
            print(f"Run {run}")
        runWeightedCurve, runOrdinaryCurve = OffPolicyFVBJ(nIters)
        weightedCurves.append(runWeightedCurve)
        ordinaryCurves.append(runOrdinaryCurve)
    return weightedCurves, ordinaryCurves

In [ ]:
WOPMCPtestStateBJ, OOPMCPtestStateBJ = prepareExperimentsBJ(100, 10000)

In [ ]:
fig = go.Figure()

yOOPMCP = np.mean((np.stack(OOPMCPtestStateBJ)[:, :, 0] - vBasePolicyBJ[testStateBJ])**2, axis=0)
fig.add_trace(go.Scatter(x=np.arange(len(yOOPMCP)), y=yOOPMCP, name="Ordinary importance sampling"))

yWOPMCP = np.mean((np.stack(WOPMCPtestStateBJ)[:, :, 0] - vBasePolicyBJ[testStateBJ])**2, axis=0)
fig.add_trace(go.Scatter(x=np.arange(len(yOOPMCP)), y=yWOPMCP, name="Weighted importance sampling"))

fig.update_xaxes(type="log")
fig.show()

#### Example 5.5 : Infinite Variance

In [ ]:
nStatesIV = 1
nActionsIV = 2
actionMappingIV = [[0, 1]] # 0 is left and 1 is right
legalActionsIV = np.array([[True, True]])
modelIV = nStatesIV, nActionsIV, legalActionsIV, actionMappingIV

In [ ]:
def sampleIVTrajectory(policy, startingState=None, startingAction=None):
    stateTrajectory = []
    actionTrajectory = []
 
    if startingAction == 1:
        stateTrajectory += [0]
        actionTrajectory += [1]
        reward = 0
 
    if startingAction == 0:
        stateTrajectory += [0]
        actionTrajectory += [0]
        nextState = np.random.choice([0, 1], p=[0.9, 0.1])
        if nextState == 1:
            reward = 1
 
    if startingAction is None or (startingAction == 0 and nextState == 0):
        nextAction = np.random.choice([0, 1], p=policy)
 
        while nextAction == 0:
            stateTrajectory += [0]
            actionTrajectory += [0]
 
            nextState = np.random.choice([0, 1], p=[0.9, 0.1])
            if nextState == 1:
                break
 
            nextAction = np.random.choice([0, 1], p=policy)
 
        if nextAction == 1:
            stateTrajectory += [0]
            actionTrajectory += [1]
            reward = 0
 
        else:
            reward = 1
 
 
    trajectory = np.empty((len(stateTrajectory), 3), dtype=int)
    trajectory[:, 0] = stateTrajectory
    trajectory[:, 1] = actionTrajectory
    trajectory[:, 2] = [0] * (len(stateTrajectory) - 1) + [reward]
 
    return trajectory

In [ ]:
def offPolicyMCPVI(gamma, maxIter, targetPolicy, EV=False, learningCurve=None):
    Q = np.zeros(2)
    nVisits = np.zeros(2)
    behaviourPolicy = np.array([0.5, 0.5])

    iter = 0 
    if learningCurve is not None:
        learningCurve[0] = np.sum(targetPolicy * Q)
 
    while iter < maxIter:
        if iter % (maxIter // 5) == 0:
            print(f"Iteration {iter}")
         
        trajectory = sampleIVTrajectory(behaviourPolicy)
         
        weight = 1.0
        gain = 0.0
        T = len(trajectory)
        for t in range(T - 1, -1, -1):
            gain = gamma * gain + trajectory[t, 2]
            if t == 0 or (t == T - 1 and trajectory[0, 1] != trajectory[T - 1, 1]) or EV:
                currentAction = trajectory[t, 1]
                nVisits[currentAction] += 1
                Q[currentAction] += (weight * gain - Q[currentAction]) / nVisits[currentAction]
            weight *= targetPolicy[trajectory[t, 1]] / behaviourPolicy[trajectory[t, 1]]
 
        iter += 1
        if learningCurve is not None and iter < maxIter:
            learningCurve[iter] = np.sum(targetPolicy * Q)
 
    return Q

In [ ]:
gammaIV = 1
basePolicyIV = np.array([1, 0])
OOPMCPtestStateIV = []
nItersIV = 100000
for run in range(10):
    print(f"Run {run}")
    runOrdinaryCurve = np.empty(nItersIV)
    offPolicyMCPVI(gammaIV, nItersIV, basePolicyIV, learningCurve=runOrdinaryCurve)
    OOPMCPtestStateIV.append(runOrdinaryCurve)

In [ ]:
fig = go.Figure()
for run in range(10):
    fig.add_trace(go.Scatter(x=np.arange(nItersIV), y=OOPMCPtestStateIV[run]))
fig.update_xaxes(type="log")
fig.update_layout(showlegend=False)
fig.show()


In [ ]:
OOPEVMCPtestStateIV = []
nItersIV = 100000
for run in range(10):
    print(f"Run {run}")
    runOrdinaryCurve = np.empty(nItersIV)
    offPolicyMCPVI(gammaIV, nItersIV, basePolicyIV, EV=True, learningCurve=runOrdinaryCurve)    
    OOPEVMCPtestStateIV.append(runOrdinaryCurve)

In [ ]:
fig = go.Figure()
for run in range(10):
    fig.add_trace(go.Scatter(x=np.arange(nItersIV), y=OOPEVMCPtestStateIV[run]))
fig.update_xaxes(type="log")
fig.update_layout(showlegend=False)
fig.show()